# Dataset & Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Pytorch

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.autograd import Variable

# CAR DATASET

In [ ]:
import os
data_dir = '/kaggle/input/cars-dataset/Cars Dataset'
os.listdir(data_dir)

# Data Transformation & DataLoader Setup

In [ ]:
from torchvision import transforms
from torchvision import datasets
import torch.utils.data as data


train_path = os.path.join(data_dir, 'train')
test_path = os.path.join(data_dir, 'test')
TRANSFORM_IMG = transforms.Compose([
    transforms.CenterCrop(112),
    transforms.Resize(112),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225] )
    ]
)
car_train_set = datasets.ImageFolder(train_path,TRANSFORM_IMG)
car_test_set = datasets.ImageFolder(test_path,TRANSFORM_IMG)

train_data_loader = data.DataLoader(car_train_set, batch_size = 8, shuffle = True)
train_data_loader_1 = data.DataLoader(car_train_set, batch_size = 8)
test_data_loader = data.DataLoader(car_test_set, batch_size = 4)


# Pytorch Logistic Regression Model

In [ ]:
#thank you very much https://www.kaggle.com/mburakergenc/ttianic-minimal-pytorch-mlp
class SimpleCNNNet(nn.Module):
    def __init__(self, batch_size = 32):
        super(SimpleCNNNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 3, 3, stride=1, padding=1) # 3x112x112
        self.conv2 = nn.Conv2d(3, 6, 3, stride=2, padding=1) # 6x56x56
        self.conv3 = nn.Conv2d(6, 12, 3, stride=2, padding=0) # 12x27x27
        self.maxpool = nn.MaxPool2d(3, stride=4)
        self.fc1 = nn.Linear(588, 7)
#         self.dropout = nn.Dropout(0.2)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.conv3(x)
        x = F.relu(x)
        x = self.maxpool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        
        return x
    
def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)
    elif isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform_(m.weight)
model = SimpleCNNNet()
model.apply(init_weights)
print(model)

# Pytorch Loss Function (Cross Entropy CE)

In [ ]:
criterion = nn.CrossEntropyLoss()

# Pytorch Optimizer (Stochastic Gradient Descent SGD)

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.003)

# Check GPU Availability

In [ ]:
from tqdm import tqdm

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Car Model Training

In [ ]:
# Parameters
batch_size = 8
n_epochs = 50

# Setup for logging
train_loss = 0
train_loss_min = np.Inf
val_loss = 0
val_loss_min = np.Inf
best_accuracy = 0.0
list_accuracies = []
# Training loop
for epoch in range(n_epochs):
    # Set model to training mode
    model.train()
    epoch_loss = 0.0
    for i, batch in tqdm(enumerate(train_data_loader), total=len(train_data_loader), desc = f"Epoch [{epoch+1}/{n_epochs}]"):
        input_img = batch[0].to(device)
        input_label = batch[1].to(device)
        # Zero the parameter gradients
        optimizer.zero_grad()
        # Forward pass 
        output_label = model(input_img)
        loss = criterion(output_label, input_label)
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Update training loss
        values, labels = torch.max(output_label, 1)
        train_loss += loss.item()
        epoch_loss += loss.item()
    train_loss = train_loss / len(car_train_set)
    if train_loss < train_loss_min:
        train_loss_min = train_loss
    
    # Validation loop
    model.eval()
    correct = 0
    total = 0
    # No gradient tracking needed, since we're not training
    with torch.no_grad():
        for inputs, labels in train_data_loader_1:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    list_accuracies.append(accuracy)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch [{epoch+1}/{n_epochs}], Train Loss: {epoch_loss:.4f} - Lowest Train Loss: {train_loss_min:.4f}")
    print(f"Validation Accuracy: {accuracy:.2f}% - Best Accuracy: {best_accuracy:.2f}%")

    torch.save(model.state_dict(), 'latest_model.pth')
print("Training completed!")


In [ ]:
best_state_dict = torch.load('/kaggle/working/best_model.pth')
# latest_state_dict = torch.load('/kaggle/working/latest_model.pth')
model_2 = SimpleCNNNet().to(device)
model_2.load_state_dict(best_state_dict)

# Car Model Testing

In [ ]:
# load model checkpoint
state_dict = torch.load('/kaggle/working/best_model.pth')
batch_size = 8
correct = 0
input_brand, output_brand = [], []

for i,batch in tqdm(enumerate(test_data_loader), total = len(test_data_loader)):
    input_img = batch[0].to(device)
    input_label = batch[1].to(device)
    # Zero the parameter gradients
    optimizer.zero_grad()
    # Forward pass 
    output = model_2(input_img)
#     print(output.shape)
    _, output_label = torch.max(output, 1)
    correct += (output_label == input_label).sum().item()
#     print(output_label.shape)
    input_brand.append(input_label)
    output_brand.append(output_label)

total = [x.shape[0] for x in output_brand]
print(total)
total = np.sum(total)
accuracy = correct / total * 100

# print('Validation accuracy = ', acc(input_brand, output_brand))    
print("Validation accuracy = ", accuracy, "%")
    
print("Testing completed!")

# Input, Output Parsing

In [ ]:
parsed_input_brand = []
parsed_output_brand = []
for arr in input_brand:
    parsed_input_brand.extend(list(arr.cpu().numpy()))
for arr in output_brand:
    parsed_output_brand.extend(list(arr.cpu().numpy()))
print(len(parsed_input_brand), len(parsed_output_brand))

# Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cnf_matrix = confusion_matrix(parsed_input_brand, parsed_output_brand)
print('Confusion matrix:')
print(cnf_matrix)

# Epoch Accuracy during Training

In [ ]:
x = np.linspace(0, 50, 50)
y = list_accuracies
plt.plot(x, y, color = 'r')
plt.show()

# Essential Layers in Deep Learning examples

In [ ]:
# Define the input tensor 
input_tensor = torch.tensor( 
    [ 
        [1, 1, 2, 4], 
        [5, 6, 7, 8], 
        [3, 2, 1, 0], 
        [1, 2, 3, 4] 
    ], dtype = torch.float32) 
  
# Reshape the input_tensor 
input_tensor = input_tensor.reshape(1, 1, 4, 4) 
  
# Initialize the Max-pooling layer with kernel 2X2 and stride 2 
pool = nn.MaxPool2d(kernel_size=2, stride=2) 
  
# Apply the Max-pooling layer to the input tensor 
output = pool(input_tensor) 
  
# Print the output tensor 
print(input_tensor)
print(output)

In [ ]:
# pool of square window of size=3, stride=2
m = nn.MaxPool2d(3, stride=2)
# pool of non-square window
m = nn.MaxPool2d((3, 2), stride=(2, 1))
input = torch.randn(20, 16, 50, 32)
output = m(input)
output

In [ ]:
# With Learnable Parameters
m = nn.BatchNorm2d(100)
# Without Learnable Parameters
m = nn.BatchNorm2d(100, affine=False)
input = torch.randn(20, 100, 35, 45)
output = m(input)
print(output)

In [ ]:
m = nn.Softmax(dim=1)
input = torch.randn(2, 3)
output = m(input)
output